# Predicting Sportsbook Point Spreads — Extended Project
## Building an Independent Power-Rating Model for NBA Betting Markets

**Skeleton Notebook** — instructions and structure only. Fill in every `# TODO` cell yourself.

### Project brief
You're a quant analyst at a sports-analytics shop. Your job is to build a model that estimates the **"fair" point spread** for an NBA game purely from team performance data (records, scoring stats, rest, injuries) — independent of what the sportsbooks are already quoting. If your model's estimate disagrees meaningfully with the market's **opening** line, that disagreement is a candidate "value bet" worth flagging for further review. You're given `sports_betting_odds.csv` (1,400 games, 28 raw columns) scraped from betting-market records.

This extended project mirrors a full quant-research workflow: a data-quality audit, domain-derived features (net rating, rest/injury advantages, sharp-vs-public money divergence), a **leakage audit** specific to betting data (the market's own closing lines are *almost the same number* as what you're trying to predict — using them as features would be cheating), several competing model families, hyperparameter tuning, feature importance, and finally a backtest of a simple betting strategy against actual game outcomes.

> **This is a synthetic dataset built for a machine-learning exercise.** The teams, stats, and outcomes are simulated, not real historical games. Nothing in this notebook is betting advice — treat the "backtest" section as a lesson in evaluating a model against a real-world decision rule, not a strategy to deploy with real money. If sports betting is a source of stress for you or someone you know, the National Council on Problem Gambling helpline (1-800-522-4700) is free and confidential.

### How to use this notebook
Each section has a short **Context** explaining *why* the step matters, then a **Task** list of exactly what to build. Use the **Cheat Sheet** notebook for syntax help and the **Background Theory** notebook for conceptual grounding. Don't peek at the Solutions notebook until you've attempted each section yourself.

### Success criteria for the whole project
- A cleaned, fully numeric feature matrix with no leakage between train and test — **and no leakage from the betting market's own closing-line data into your features**
- At least 3 trained regression models, fairly compared on the same test split
- A tuned final model with cross-validated performance estimates
- Two independent feature-importance views that agree on the top spread drivers
- A backtest comparing your model's disagreements with the market against actual game outcomes, with clearly labeled synthetic-data caveats


## Module 0 — Environment Setup

**Context.** Fixed random seeds and consistent imports make your results directly comparable to the Solutions notebook.

**Task**
- Import `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`.
- Import `train_test_split`, `cross_val_score` from `sklearn.model_selection`.
- Import `LinearRegression`, `Ridge` from `sklearn.linear_model`.
- Import `RandomForestRegressor`, `GradientBoostingRegressor` from `sklearn.ensemble`.
- Import `mean_absolute_error`, `mean_squared_error`, `r2_score` from `sklearn.metrics`.
- Set `RANDOM_STATE = 42` and use it everywhere a `random_state` argument exists.


In [ ]:
# TODO: imports and global constants


## Module 1 — Data Loading & Initial Inspection

**Context.** Betting data mixes team-level identifiers, market quotes, and scraped text fields — get the shape of the problem before touching anything.

**Task**
- Load `sports_betting_odds.csv` into `df`.
- Print `df.shape`, `df.head()`, `df.info()`.
- Print `df.describe(include='all').T`.
- List which columns are numeric-looking but stored as text (hint: look for `"pts"`, `"%"`, `"$"`, `"days"`, `"miles"`, and win-loss records like `"42-18"`).


In [ ]:
# TODO: load data and inspect


## Module 2 — Data Quality Audit

**Context.** Scraped betting data has its own flavor of messiness: `"Not Reported"` injury counts, `"Unknown"` travel distances, and inconsistent sportsbook name casing (`"draftkings"` vs `"DraftKings"` vs `"DRAFTKINGS"`).

**Task**
- Build a dtype/nunique/missing audit table.
- Check `df.duplicated().sum()`; inspect a few duplicated rows before deciding whether to drop them.
- Print `.unique()` for every text column and flag disguised-missing sentinels.
- Normalize `sportsbook` casing so the four real sportsbooks aren't split into 8+ pseudo-categories.


In [ ]:
# TODO: dtype audit table


In [ ]:
# TODO: duplicate check + per-column unique-value scan + sportsbook casing fix


## Module 3 — Feature Engineering I: Records & Scoring Stats

**Context.** `home_record`/`away_record` are `"W-L"` strings — you need a win percentage, not two counting numbers glued together with a hyphen. `home_ppg`/`away_ppg`/`home_papg`/`away_papg` are numeric quantities trapped behind a `" pts"` suffix.

**Task**
- Write a helper that converts a `"W-L"` string into a win percentage (handle the 0-games-played edge case).
- Create `home_win_pct`, `away_win_pct`, `home_games_played`, `away_games_played`; drop the original record columns.
- Strip `" pts"` from `home_ppg`, `away_ppg`, `home_papg`, `away_papg` and cast to `float`.


In [ ]:
# TODO: parse win-loss records into win percentages


In [ ]:
# TODO: clean ppg/papg columns


## Module 4 — Feature Engineering II: Rest, Injuries, Travel & Market Context

**Context.** Several columns need unit-stripping (`" days"`, `"%"`, `"$...M"`), and two columns (`home_injuries`/`away_injuries`, `away_travel_distance`) have a **disguised-missing sentinel** (`"Not Reported"`, `"Unknown"`) mixed in with otherwise-numeric text.

**Task**
- Strip `" days"` from `home_rest_days`/`away_rest_days` and cast to `int`.
- Clean `home_injuries`/`away_injuries`: strip `" players"`, replace `"Not Reported"` with a sentinel, cast to `int`, then **create a `*_unreported` flag column** and impute the sentinel rows with the column's median (excluding the sentinel).
- Clean `away_travel_distance`: strip the thousands-comma and `" miles"`, treat `"Unknown"` as missing, create a `travel_unknown` flag, and impute with the median.
- Strip `"%"` from `public_bet_pct_home`, `sharp_bet_pct_home`, `venue_capacity_pct` and cast to `int`.
- Strip `"$"` and `"M"` from `betting_volume` and cast to `float`.
- Convert `home_back_to_back`, `away_back_to_back`, `home_star_out`, `away_star_out` from `"Yes"/"No"` to `0/1`.


In [ ]:
# TODO: rest days, injuries (+ missing flags), travel distance (+ missing flag)


In [ ]:
# TODO: percentage columns, betting volume, Yes/No flags


## Module 5 — Feature Engineering III: Domain-Derived Features

**Context.** Raw per-team stats only go so far — what actually drives a point spread is the **difference** between the two teams, plus market-sentiment signals unique to betting data.

**Task** — engineer at least six of the following:
- `net_rating_home` / `net_rating_away` (ppg − papg for each team) and `net_rating_diff`.
- `win_pct_diff` (home − away).
- `rest_advantage` (home rest days − away rest days).
- `injury_advantage` (away injuries − home injuries — more injuries on the other team favors you).
- `star_out_diff` (away star-out flag − home star-out flag).
- `sharp_public_divergence` (sharp money % − public money % on the home side) — a classic "smart money vs. public money" signal.
- Justify each new feature in one sentence: what does it capture that the raw per-team columns don't state directly?


In [ ]:
# TODO: domain-derived differential features (at least 6)


## Module 6 — Advanced EDA & the Betting-Data Leakage Audit

**Context.** This is the step that's genuinely different from a typical tabular project: betting datasets contain **multiple market quotes for the same underlying event**. `opening_spread_home` and `closing_moneyline_home` are both extremely tightly correlated with your target (`closing_spread_home`) — because they're all just different snapshots/formats of the *same* market efficiency. Including them as model inputs would let the model "predict" the closing spread by nearly just reading off a number that already encodes it, defeating the entire purpose of building an *independent* rating model.

**Task**
- Plot the distribution of `closing_spread_home`; compute its skewness.
- Build a correlation heatmap on the numeric features **including** `opening_spread_home` and `closing_moneyline_home`. Note how extreme their correlation with the target is compared to every team-performance feature.
- Explicitly compute `df['closing_spread_home'].corr(df['opening_spread_home'])` and `df['closing_spread_home'].corr(df['closing_moneyline_home'])`. Write one paragraph explaining, in your own words, why using either as a model feature would be a form of leakage **relative to this project's specific goal** (even though `opening_spread_home` is technically known before the game starts).
- Decide — and justify in writing — which columns to exclude from the feature set before modeling.
- Compute VIF on your remaining numeric features and flag anything above 10.
- Boxplot `closing_spread_home` by `home_back_to_back` and by `home_star_out`.
- Run an IQR-based outlier check on `closing_spread_home`; decide whether to keep flagged rows.


In [ ]:
# TODO: target distribution + skew


In [ ]:
# TODO: correlation heatmap INCLUDING the market-quote columns


In [ ]:
# TODO: explicit correlation check + written leakage decision


In [ ]:
# TODO: VIF check on the features you plan to keep


In [ ]:
# TODO: boxplots + IQR outlier check


## Module 7 — Encoding Categorical Variables (Leakage-Safe)

**Context.** `home_team`/`away_team` (30 teams each) are high-cardinality categoricals; `sportsbook` (4 values, post-cleanup) is low-cardinality. Team identity should be target-encoded (fit on train only) since one-hot-encoding 30+2 columns for two team slots would be unwieldy and sparse.

**Task**
- Split remaining categorical columns into low-cardinality (one-hot, `drop_first=True`) and high-cardinality (target-encode) using the same `< 5 unique values` threshold as before.
- Split into train/test **before** computing any target-encoding statistic.
- Fit target-encoding means for `home_team` and `away_team` using `y_train`/`X_train` only; apply to `X_test`; fall back to the training global mean for any team-role combination unseen in training.
- Confirm zero `NaN`s remain in `X_train`/`X_test`.


In [ ]:
# TODO: bucket categoricals by cardinality


In [ ]:
# TODO: train/test split, then fit + apply leakage-safe target encoding


## Module 8 — Baseline & Linear Models

**Context.** Establish the floor before reaching for anything fancy — and note that if the excluded market-quote columns really do carry almost all the signal, your remaining model will have noticeably higher error than a (leaky) model that used them. That's expected and is the whole point.

**Task**
- Build a mean-predictor baseline; report MAE/RMSE/R² on the test set.
- Fit `LinearRegression`; report the same three metrics.
- Fit `Ridge`; compare coefficients to plain linear regression.


In [ ]:
# TODO: mean-predictor baseline


In [ ]:
# TODO: LinearRegression, Ridge


## Module 9 — Tree-Ensemble Models

**Context.** Point spreads in this dataset are generated from a mostly *additive* combination of team-strength differentials — don't assume tree ensembles will automatically win here the way they did on a more interaction-heavy problem. Let the numbers decide.

**Task**
- Train `RandomForestRegressor(random_state=RANDOM_STATE)` with default hyperparameters.
- Train `GradientBoostingRegressor(random_state=RANDOM_STATE)` with default hyperparameters.
- Collect all models trained so far (baseline, linear, ridge, RF, GB) into one comparison table sorted by MAE.


In [ ]:
# TODO: RandomForestRegressor, GradientBoostingRegressor


In [ ]:
# TODO: model comparison table


## Module 10 — Hyperparameter Tuning & Cross-Validation

**Context.** A single 80/20 split is one noisy draw — cross-validation gives you a distribution of performance estimates, not just a point estimate.

**Task**
- Run 5-fold `cross_val_score` (scoring `'neg_mean_absolute_error'`) on your best model from Module 9. Report mean ± std.
- Define a hyperparameter search space appropriate to your best model family.
- Run `RandomizedSearchCV` (`cv=5`, `scoring='neg_mean_absolute_error'`).
- Report best params, best CV score, and the refit model's test-set performance.


In [ ]:
# TODO: cross_val_score on your leading candidate


In [ ]:
# TODO: RandomizedSearchCV, refit, evaluate on test set


## Module 11 — Evaluation & Diagnostics

**Context.** A spread-prediction model's errors matter differently at the extremes — a 2-point miss on a near-pick'em game is a bigger practical error than a 2-point miss on a 15-point blowout line.

**Task**
- Report MAE, RMSE, R², and MAPE for your final model (careful: MAPE is unstable near `closing_spread_home ≈ 0` — note this in your writeup rather than silently trusting the number).
- Plot predicted vs. actual spread with a `y=x` reference line.
- Plot residuals vs. predicted values; check for heteroscedasticity.
- Note where the model over/underestimates.


In [ ]:
# TODO: MAE, RMSE, R2, MAPE (with a note on MAPE's instability here)


In [ ]:
# TODO: predicted vs actual scatterplot + residual plots


## Module 12 — Feature Importance

**Context.** Confirming which factors the model actually leans on is essential before trusting it enough to compare against the market.

**Task**
- Extract and plot the top 10 features by impurity-based importance (if using a tree model) or by absolute coefficient value (if your final model is linear).
- Compute and plot permutation importance on the test set.
- Compare the two — do `net_rating_diff` and `win_pct_diff` dominate, as domain knowledge would predict? If not, investigate why.


In [ ]:
# TODO: importance plot #1


In [ ]:
# TODO: permutation importance plot


## Module 13 — Backtest: Does Disagreeing With the Market Pay Off?

**Context.** This is the sports-betting-specific payoff of the whole exercise. If your model's fair-spread estimate disagrees with the market's **opening** line by more than some threshold, that's a candidate signal. Backtesting means checking, using the actual game outcome (`home_margin`, held out of your features the whole time), whether betting on that signal would have won more often than chance.

> Remember: this is a synthetic dataset built for a machine-learning exercise, not real historical data or betting advice.

**Task**
- For your test-set games, compute `edge = model_prediction - opening_spread_home`.
- Filter to games where `abs(edge)` exceeds a threshold you choose (e.g. 2.0 points); decide the recommended bet side from the sign of `edge`.
- Using `home_margin` and `closing_spread_home`, determine whether each recommended bet would have won "against the spread" (ATS).
- Compute the strategy's win rate and simulate ROI at standard -110 odds (a win pays +0.909, a loss costs -1.0).
- Compare against the baseline of betting every single game, and against the break-even win rate needed at -110 odds (52.4%).
- Write two sentences interpreting the result, including at least one reason to be skeptical of a good-looking backtest on a small/synthetic sample.


In [ ]:
# TODO: compute edge, filter to threshold, determine bet side


In [ ]:
# TODO: score ATS outcomes, compute win rate + simulated ROI, compare to baseline


## Module 14 — Persistence & Inference

**Context.** A model that only lives in a notebook variable can't be run against next week's games.

**Task**
- Save your final model with `joblib.dump`, along with encoding maps and the training column order.
- Write a function `predict_fair_spread(raw_game_dict)` that takes a dictionary shaped like one row of the raw CSV (minus the market-quote and outcome columns) and returns your model's fair-spread estimate.
- Test it on 2–3 made-up matchups and sanity-check the outputs are plausible spreads.


In [ ]:
# TODO: persist model + encoders


In [ ]:
# TODO: predict_fair_spread(raw_game_dict) function + sanity checks


## Module 15 — Conclusions & Write-Up

**Task**
- Write a 150–250 word summary covering: which model you chose and why, its expected error margin in points, the top 3–5 spread drivers, the backtest result and at least one reason to interpret it cautiously, and one concrete next step.


*(Write your conclusions here.)*